# W05B A · 해석·척도·추정량

2026-09-30 · **R 커널** · 런타임→모두 실행. 연결이 끊기면 준비 셀부터 다시 실행하세요.

![같은 조건을 다른 척도 또는 구간 대상으로 비교한 수업 그림](https://raw.githubusercontent.com/lunalab-ai/regre/2026-fall-w05b/course/notion/w05b/assets/a02-centering.png)

수업의 R 계산으로 만든 그림입니다. 출력이 저장되어 있어도 현재 객체가 살아 있다는 뜻은 아닙니다.


**R 실습**입니다. 온라인 R 또는 R Colab A/B 중 한 환경을 선택하세요. 먼저 예상하고, 실행한 뒤 단위와 비교 조건을 포함해 설명합니다. 학생 연습 셀은 주석으로 되어 있어 기본 흐름을 막지 않습니다. `#`를 지우고 필요한 부분을 완성하세요. 데이터는 R 내장 attitude로 자동 준비되며 파일 업로드·API 키·패키지 설치가 필요 없습니다. 다운로드 R/QMD는 data 폴더를 함께 보관하세요.

공통 소스는 고정 버전 **2026-fall-w05b**입니다. `source()`는 정의를 읽고, `lm()`은 모형을 적합하며, 표와 구간 함수는 그 적합 결과를 읽습니다. [API 전체 안내](https://github.com/lunalab-ai/regre/blob/2026-fall-w05b/src/mlr-inference-api.md). 세션을 재시작하면 준비 셀부터 다시 실행하세요.


In [ ]:
# 공개 고정 버전과 동일한 소스만 사용합니다. MD5는 파일 동일성 확인용입니다.
source_url <- "https://raw.githubusercontent.com/lunalab-ai/regre/2026-fall-w05b/src/mlr-inference.R"
expected_md5 <- "220bb19be470b1d35e13f48ed562ab96"
local_candidates <- c("src/mlr-inference.R", "../../src/mlr-inference.R")
existing <- local_candidates[file.exists(local_candidates)]
if (length(existing)) {
  helper_path <- existing[1]; helper_origin <- "동봉 소스"
} else {
  cache_dir <- file.path(path.expand("~"),".cache","regre","2026-fall-w05b")
  dir.create(cache_dir,recursive=TRUE,showWarnings=FALSE)
  helper_path <- file.path(cache_dir,"mlr-inference.R")
  helper_origin <- if(file.exists(helper_path)) "검증할 캐시" else "고정 버전 최초 다운로드"
  options(timeout=120)
  if (!file.exists(helper_path)) {
    download_status <- download.file(source_url,helper_path,mode="wb",quiet=FALSE)
    if(download_status!=0) stop("공통 소스 다운로드 실패: 인터넷을 확인하고 준비 셀을 다시 실행하세요.")
  }
}
if (unname(tools::md5sum(helper_path))!=expected_md5)
  stop("소스 해시 불일치: 변경한 소스를 별도 보관한 뒤 고정 버전 파일을 다시 받으세요.")
source(helper_path)
cat(helper_origin,"|",R.version.string,"\n")
d <- datasets::attitude
fit2 <- lm(rating~complaints+learning,data=d)
fit6 <- lm(rating~.,data=d)
stopifnot(nrow(d)==30,df.residual(fit2)==27)

## 1. 같은 데이터, 다른 두 변수 모형

한 행은 부서이고 값은 긍정 응답 비율(%)입니다. 지난 차시 complaints+privileges와 오늘 complaints+learning을 구별합니다. 다른 변수가 같은 조건의 차이를 읽으세요.


In [ ]:
head(d,3)
coef(fit2)
conditions <- data.frame(complaints=c(60,70),learning=c(55,55))
predict(fit2,newdata=conditions)

### L1 · 예측–실행–설명

learning=55로 고정하고 complaints를 60에서 65로 바꾼 예측 차이를 구하세요. 계수와 단위를 포함해 문장으로 쓰세요.


In [ ]:
# new_rows <- data.frame(complaints=c(60,65),learning=c(55,55))
# change <- ...
# change

## 2. 중심화와 원래 예측 비교

[mlr_scaling 정의](https://github.com/lunalab-ai/regre/blob/2026-fall-w05b/src/mlr-inference.R#L101)는 rating/complaints/learning 숫자 열을 받아 네 모형과 계수표·원래 단위 복원 예측·학습 중심/척도를 반환합니다. 원본 자료를 바꾸지 않습니다. 예: `mlr_scaling(d)$coefficients`. 절편 포함 모형이며 상수 열과 결측은 거부합니다.


In [ ]:
scaling <- mlr_scaling(d)
scaling$coefficients
sapply(scaling$fitted_original_units,function(x) max(abs(x-fitted(fit2))))
c(SD_z=sd(scaling$standardized$complaints),
  sum_z2=sum(scaling$standardized$complaints^2),
  sum_unit2=sum(scaling$unit_length$complaints^2))

### L2 · 예측–실행–설명

X만 중심화한 모형의 절편·기울기·SSE를 원래 모형과 비교하세요. Y까지 표준화한 예측은 어떤 변환 뒤 비교해야 하나요?


In [ ]:
# centered <- scaling$models$centered
# c(original=deviance(fit2),centered=...)
# restored <- ...

## 3. 새 자료를 학습 좌표로 옮기기

새 부서 한 행에 `scale()`을 다시 하면 분산을 구할 수 없습니다. 학습 때 저장한 평균·표준편차를 씁니다. 기본 R의 [scale 문서](https://stat.ethz.ch/R-manual/R-devel/library/base/html/scale.html)에서 center=FALSE일 때의 정의도 읽으세요.


In [ ]:
training_center <- scaling$centers[c("complaints","learning")]
training_scale <- scaling$scales[c("complaints","learning")]
new_raw <- data.frame(complaints=70,learning=60)
training_center
training_scale

### L3 · 예측–실행–설명

오류 생각 실험: `scale(new_raw)`로 새 행을 변환하려던 코드를 고치세요. 학습 좌표로 옮긴 예측을 원래 단위로 복원하면 직접 예측과 같은지 검사하세요.


In [ ]:
# new_z <- as.data.frame(sweep(sweep(as.matrix(new_raw),2,...,"-"),2,...,"/"))
# pred_back <- ...

## 4. 반복 표본: 오차 규모와 계수의 흔들림

[mlr_simulate 정의](https://github.com/lunalab-ai/regre/blob/2026-fall-w05b/src/mlr-inference.R#L130)는 고정 X, 참 계수(10,.65,.20), 독립 정규오차로 B개 모의 자료를 적합합니다. B=1000, sigma_error=7, seed=730이 기본입니다. draws는 B×3 계수표, summary는 이론·경험 표준편차입니다. 실제 rating을 다시 수집하지 않으며 난수 상태를 복원합니다.


In [ ]:
simulation <- mlr_simulate(B=1000,sigma_error=7,seed=730)
simulation$summary
hist(simulation$draws$complaints,breaks=25,col="#9ed3d5",
     main="Fixed X, new Gaussian errors",xlab="Estimated complaints slope")
abline(v=.65,col="#d95d52",lwd=3)

### L4 · 예측–실행–설명

같은 seed에서 오차 표준편차를 14로 바꾸세요. 계수 분포의 중심과 폭은 각각 어떻게 달라질까요? 추정 오차와 표준오차를 구분해 설명하세요.


In [ ]:
# larger_noise <- mlr_simulate(B=1000,sigma_error=...,seed=730)
# larger_noise$summary

## 5. R, R²와 수정 R²

y와 ŷ의 상관과 전체 손실 감소를 비교합니다. 절편 포함 OLS이며 두 모형 모두 동일한 30행을 씁니다. 평균만 모형은 ŷ가 상수이므로 상관계수식의 예외입니다.


In [ ]:
fit_comparison <- do.call(rbind,lapply(list(two=fit2,six=fit6),function(m)
  data.frame(SSE=deviance(m),df=df.residual(m),R=cor(d$rating,fitted(m)),
             R2=summary(m)$r.squared,adjusted_R2=summary(m)$adj.r.squared)))
fit_comparison
plot(d$rating,fitted(fit2),pch=19,col="#087f8c",xlab="Observed rating",ylab="Fitted rating")
abline(0,1,lty=2)
stopifnot(abs(cor(d$rating,fitted(fit2))^2-summary(fit2)$r.squared)<1e-10)

In [ ]:
stopifnot(nrow(d)==30,df.residual(fit2)==27,max(abs(scaling$fitted_original_units$original-scaling$fitted_original_units$standardized))<1e-10,abs(summary(fit2)$r.squared-.7080152)<1e-6,nrow(simulation$draws)==1000)

다음은 [Colab B: 검정·예측](https://colab.research.google.com/github/lunalab-ai/regre/blob/2026-fall-w05b-r1/notebooks/student/w05b_inference.ipynb)입니다. 마지막에는 [누적 웹 앱](https://lunalab-ai.github.io/regre/apps/w05b/edit/index.html)의 입력→계산→출력을 직접 수정합니다.